# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
!git clone https://github.com/nandini3206/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 132 (delta 45), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (132/132), 1.87 MiB | 7.33 MiB/s, done.
Resolving deltas: 100% (45/45), done.


In [11]:
%cd flyrank-ml-internship
!pwd

/content/flyrank-ml-internship
/content/flyrank-ml-internship


In [12]:
!ls

AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [13]:
from IPython.display import Markdown, display

display(Markdown("""
## Method Choice

This project predicts which pages should be refreshed based on historical page signals.

Since the target is a binary classification problem, multiple supervised learning models are compared.

Models selected:

- Logistic Regression
- Decision Tree
- Random Forest

Reason:

- Logistic Regression provides a simple and interpretable baseline.
- Decision Tree captures nonlinear relationships between page features.
- Random Forest improves predictive performance by combining multiple decision trees and reducing overfitting.

The final model will be selected based on Precision@50 because the goal is to rank the highest-priority pages for refresh.
"""))


## Method Choice

This project predicts which pages should be refreshed based on historical page signals.

Since the target is a binary classification problem, multiple supervised learning models are compared.

Models selected:

- Logistic Regression
- Decision Tree
- Random Forest

Reason:

- Logistic Regression provides a simple and interpretable baseline.
- Decision Tree captures nonlinear relationships between page features.
- Random Forest improves predictive performance by combining multiple decision trees and reducing overfitting.

The final model will be selected based on Precision@50 because the goal is to rank the highest-priority pages for refresh.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [14]:
from IPython.display import Markdown, display

display(Markdown("""
## Split Design

A client-aware train/test split is used whenever possible.

This prevents pages from the same client appearing in both training and testing datasets.

If a client-based split cannot be created, a stratified split is used while preserving class balance.

This produces an honest evaluation and avoids information leakage.
"""))


## Split Design

A client-aware train/test split is used whenever possible.

This prevents pages from the same client appearing in both training and testing datasets.

If a client-based split cannot be created, a stratified split is used while preserving class balance.

This produces an honest evaluation and avoids information leakage.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [15]:
!ls scripts

01_prepare_features.py	04_evaluate_and_export.py  run_all.py
02_baseline_score.py	05_build_pdf_report.py
03_train_model.py	ml_utils.py


In [16]:
!ls outputs

charts	model_report.md  refresh_queue_sample.csv


In [17]:
!sed -n '1,200p' scripts/03_train_model.py

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from ml_utils import (
    MODEL_CATEGORICAL_FEATURES,
    MODEL_NUMERIC_FEATURES,
    OUTPUT_DIR,
    PROCESSED_DIR,
    display_path,
    precision_at_k,
    write_json,
)


FEATURE_PATH = PROCESSED_DIR / "refresh_feature_vector.csv"
BASELINE_PATH = PROCESSED_DIR / "baseline_refresh_queue.csv"
PREDICTION_PATH = PROCESSED_DIR / "model_predictions.csv"
RESULT_PATH = OUTPUT_DIR / "model_results.json"
RANDOM_STATE = 42


def parse_args

In [18]:
!sed -n '1,200p' scripts/ml_utils.py

from __future__ import annotations

import json
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd


ROOT = Path(__file__).resolve().parents[1]
RAW_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
OUTPUT_DIR = ROOT / "outputs"
CHART_DIR = OUTPUT_DIR / "charts"


NUMERIC_COLUMNS = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate"

In [25]:
!python scripts/02_baseline_score.py

Wrote baseline queue: /content/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340


In [26]:
!ls data/processed

baseline_metadata.json	    feature_metadata.json
baseline_refresh_queue.csv  refresh_feature_vector.csv


In [27]:
!python scripts/03_train_model.py

Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship/outputs/model_results.json


In [29]:
from IPython.display import Markdown, display
import json
import pandas as pd

with open("outputs/model_results.json", "r") as f:
    results = json.load(f)

baseline = results["baseline"]
models = results["models"]

comparison = pd.DataFrame({
    "Model": [
        "Baseline",
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Accuracy": [
        baseline["baseline_accuracy"],
        models["logistic_regression"]["accuracy"],
        models["decision_tree"]["accuracy"],
        models["random_forest"]["accuracy"]
    ],
    "Precision@50": [
        baseline["baseline_precision_at_50"],
        models["logistic_regression"]["precision_at_50"],
        models["decision_tree"]["precision_at_50"],
        models["random_forest"]["precision_at_50"]
    ],
    "ROC-AUC": [
        baseline["baseline_roc_auc"],
        models["logistic_regression"]["roc_auc"],
        models["decision_tree"]["roc_auc"],
        models["random_forest"]["roc_auc"]
    ]
})

display(comparison)

display(Markdown(f"""
### Model Comparison

The baseline model achieved a Precision@50 of **{baseline['baseline_precision_at_50']:.2f}**.

Three supervised learning models were trained and evaluated using the same client-holdout split.

The **Random Forest** model produced the best ranking performance with:

- Precision@50 = **{models['random_forest']['precision_at_50']:.2f}**
- ROC-AUC = **{models['random_forest']['roc_auc']:.2f}**
- Accuracy = **{models['random_forest']['accuracy']:.2f}**

Since the project goal is to rank pages most likely to require refresh, Random Forest was selected as the final model.
"""))

,Model,Accuracy,Precision@50,ROC-AUC
0,Baseline,0.608602,0.24,0.626892
1,Logistic Regression,0.660645,0.40,0.700291
2,Decision Tree,0.676559,0.58,0.741520
3,Random Forest,0.672258,0.74,0.750030



### Model Comparison

The baseline model achieved a Precision@50 of **0.24**.

Three supervised learning models were trained and evaluated using the same client-holdout split.

The **Random Forest** model produced the best ranking performance with:

- Precision@50 = **0.74**
- ROC-AUC = **0.75**
- Accuracy = **0.67**

Since the project goal is to rank pages most likely to require refresh, Random Forest was selected as the final model.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [31]:
from IPython.display import Markdown, display
import json

with open("outputs/model_results.json", "r") as f:
    results = json.load(f)

top_features = results["best_model"]["feature_importance_top"][:5]

text = """
### Error Analysis

The Random Forest model improves prediction quality, but some mistakes are still expected.

Possible sources of error include:

- Pages with changing user behavior over time.
- Limited historical information for some pages.
- Seasonal or unexpected traffic changes.
- Similar feature patterns between declining and healthy pages.

Top features used by the model were:
"""

for feature in top_features:
    text += f"\n- **{feature['feature']}** (importance = {feature['importance']:.3f})"

text += """

Overall, the model should be used as a decision-support system rather than a fully automatic decision maker.
"""

display(Markdown(text))


### Error Analysis

The Random Forest model improves prediction quality, but some mistakes are still expected.

Possible sources of error include:

- Pages with changing user behavior over time.
- Limited historical information for some pages.
- Seasonal or unexpected traffic changes.
- Similar feature patterns between declining and healthy pages.

Top features used by the model were:

- **days_with_impressions** (importance = 0.158)
- **log_impressions_90d** (importance = 0.129)
- **avg_position** (importance = 0.109)
- **content_age_days** (importance = 0.095)
- **char_count** (importance = 0.043)

Overall, the model should be used as a decision-support system rather than a fully automatic decision maker.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.